# 📊 Solução: Migração de Notebooks para Apache Glue
## Raciocínio Técnico e Decisões de Design

**Épico**: E5 - Analytics Exploratória  
**Autor**: Lucas Bittencourt  
**Data**: 2026-05-18  
**Objetivo**: Documentar o processo de migração de transformações de dados manualmente executadas (notebooks) para um job Apache Glue production-ready

---

## 📌 Introdução: O Desafio

### Contexto do Problema

O projeto **AIOps Locaweb** busca **prever incidentes operacionais** em uma plataforma ITSM que registra ~122k incidentes/ano. Para isso, precisamos transformar dados brutos em features ML de forma **confiável, escalável e monitorada**.

**Pergunta Central**: Como migrar transformações experimentais (notebooks) para um pipeline production-grade?

## 🏗️ Fase 1: Entender a Arquitetura Medallion Lake

### O Fluxo de Dados

```
LW-DATASET.xlsx (Local)
    ↓
S3 Bronze (raw CSV)
  └─ incidents_standardized.parquet
    ↓ [ESTE É O NOSSO FOCO - Glue Job]
S3 Silver (engineered features)
  └─ incidents_silver_2025.parquet
    ↓ [Próximas fases]
S3 Gold + RDS PostgreSQL (Star Schema + ML features)
    ↓
ML Models (Prophet, XGBoost, K-Means)
    ↓
Power BI Dashboards
```

### Camadas Explicadas

| Camada | Responsabilidade | Qualidade de Dados | Escopo |
|--------|------------------|------------------|--------|
| **Bronze** | Raw data (XLSX → CSV) | Dados como chegam (brutos) | Todos os incidentes |
| **Silver** | Limpeza, features derivadas | Validado, pronto para ML | Esforço real (pós-2025) |
| **Gold** | Star schema, agregações | Pronto para BI/ML | Dimensões + fatos |

### Por que a Arquitetura Medallion?

✅ **Separação de responsabilidades**: cada camada tem um propósito claro  
✅ **Rastreabilidade**: posso voltar ao Bronze se Silver ficar corrupto  
✅ **Escalabilidade**: Parquet particionado é eficiente em custo S3  
✅ **Reutilização**: Silver pode alimentar múltiplos consumidores (dbt, Airflow, ML)

## 🔍 Fase 2: Análise do Bronze - Entender o Input

### Metadados do Bronze

```
Input: s3://aiops-locaweb-datalake-2026/bronze/incidents_standardized.parquet
├─ Total de registros: 122.543
├─ Período: 2018-2026 (8 anos)
├─ Partições: Ano/Mês (S3)
└─ Colunas: 19 (tipos mistos: string, datetime, int, null)
```

### Colunas Originais (Do ITSM)

```python
colunas_bronze = {
    'Número': str,                    # ID único do incidente
    'Prioridade': str,                # 'P1', 'P2', 'P3', 'P4', 'P5'
    'Produto': str,                  # Categoria do sistema (pode ser nulo)
    'Categoria': str,                # Sub-classificação (pode ser nulo)
    'Subcategoria': str,              # Nível adicional (pode ser nulo)
    'Grupo_designado': str,           # Equipe responsável
    'Aberto': datetime,               # Data/hora de abertura
    'Resolvido': datetime,            # Data/hora de resolução (pode ser nulo)
    'Encerrado': datetime,            # Data/hora de encerramento (pode ser nulo)
    'Duração': int,                   # Segundos entre abertura e encerramento
    'Status': str,                    # 'Aberto', 'Resolvido', 'Encerrado', 'Sem Intervenção'
    'Entrou_para_KPI': str,           # 'SIM', 'NAO', nulo
    'KPI_Violado': str,               # 'SIM', 'NAO', nulo (target potencial)
    'Incidente_Pai': str,             # Vincuação a incidente pai (pode ser nulo)
    'Código_de_fechamento': str,      # Motivo do encerramento (pode ser nulo)
    'Solução': str,                   # Descrição da solução (pode ser nulo)
    'Aberto_por': str,                # Usuário que abriu
    'Descrição_resumida': str         # Summary do incidente
}
```

### Decisão Crítica: Qual é o "Esforço Real"?

🔑 **Problema**: ITSM registra tudo, inclusive eventos que não exigem ação (e.g., notificações, monitoramento).

💡 **Solução**: Filter por `Status != 'Sem Intervenção'`

```python
# Exemplo de ruído de monitoramento:
INC-2025-001234  # Status='Sem Intervenção' → Apenas notificação, sem esforço
INC-2025-001235  # Status='Aberto'          → Realmente requer investigação
```

**Impacto**:
- Bronze: 122.543 registros
- Pós-filtro (Status != 'Sem Intervenção'): ~41.441 registros (33% de esforço real)

### Por que também filtrar por Data (>= 2025)?

❌ **Dados históricos (2018-2024)**: Contexto operacional mudou (processos, ferramentas, equipes)  
✅ **Dados recentes (2025+)**: Representam operação atual, melhor para forecast

**Trade-off**: Menos dados, mas mais representativos

## 🎯 Fase 3: Decisões de Design - Por que Glue?

### Opções Consideradas

| Solução | Prós | Contras | Escolhida? |
|---------|------|---------|------------|
| **Notebook Jupyter** | Interativo, fácil debug | Não escalável, sem SLA, difícil versionar | ❌ |
| **Script Python + Cron** | Simples, barato | Sem logging, sem retry, sem monitoramento | ❌ |
| **Apache Spark (standalone)** | Escalável | Gerenciamento complexo, infraestrutura própria | ❌ |
| **Apache Glue (AWS)** | Managed, logging CloudWatch, retry automático, Spark | Custo pode ser alto se job de longa duração | ✅ |
| **Databricks** | Excelente UX, Unity Catalog | Custo elevado, menos flexibilidade com AWS | ❌ |

### Por que Apache Glue?

✅ **Managed Service**: AWS gerencia clusters, patching, scaling  
✅ **Integração nativa com S3**: Ótimo para data lakes  
✅ **PySpark + Python 3.9+**: Já conhecemos pandas/PySpark  
✅ **CloudWatch Logging**: Logs automáticos, fácil alertar  
✅ **IAM Roles**: Segurança declarativa (Terraform)  
✅ **Scheduling via Airflow**: Orquestração futura (E9)  
✅ **Custo-benefício**: Pago apenas pelo tempo executado (DPUs × minutos)

### Configuração do Job

```
Worker Type:      G.2X (16 vCPU, 122 GB RAM por worker)
Num Workers:      10
Total:            160 vCPU, 1.2 TB RAM
Timeout:          60 minutos
Max Retries:      1 (se falhar, tenta 1x mais)
Glue Version:     4.0 (PySpark 3.3, Python 3.9)
```

**Justificativa do tamanho**: 122k registros × ~200 features/transformações = ~25M operações. G.2X × 10 workers oferece paralelismo suficiente.

## 🔧 Fase 4: Feature Engineering - As 6 Derivadas

### Princípio de Design

**"Uma feature é válida se um modelo a consegue usar, e se seu significado é interpretável."**

Não criamos features aleatoriamente. Cada uma responde uma pergunta:

### 1️⃣ Feature: `Exige_Intervencao` (Binary Flag)

**Pergunta**: "Este incidente realmente exigiu esforço, ou foi apenas ruído de monitoramento?"

**Lógica**:
```python
Exige_Intervencao = 1 if Status != 'Sem Intervenção' else 0
```

**Uso em ML**: 
- Filter primário (treino apenas com Exige_Intervencao=1)
- Feature categórica para stratificação

**Exemplo**:
```
INC-2025-001: Status='Aberto'         → Exige_Intervencao=1 ✅
INC-2025-002: Status='Sem Intervenção' → Exige_Intervencao=0 (filtrado)
```

---

### 2️⃣ Feature: `Prioridade_Num` (Numeric)

**Pergunta**: "Qual é a prioridade do incidente em escala numérica?"

**Lógica**:
```python
Prioridade_Num = int(Prioridade[0])  # 'P1'→1, 'P2'→2, ..., 'P5'→5
```

**Uso em ML**: 
- Feature contínua para XGBoost
- Correlação com SLA (P1 tem SLA menor)

**Validação**:
```
Prioridade='P1' → Prioridade_Num=1 (SLA: 4 horas)
Prioridade='P5' → Prioridade_Num=5 (SLA: 5 dias)
```

---

### 3️⃣ Feature: `Possui_Pai` (Binary Flag)

**Pergunta**: "Este incidente está vinculado a um incidente pai?"

**Lógica**:
```python
Possui_Pai = 1 if Incidente_Pai is not null else 0
```

**Importância Regulatória**:
- Incidentes filhos herdam SLA do pai
- Violação do filho != violação do pai
- **Decisão de negócio**: Se Possui_Pai=1, o incidente NÃO conta para risco (isenção regulatória)

**Exemplo**:
```
INC-2025-MASTER (pai)
├─ INC-2025-CHILD-1 (filho) → Possui_Pai=1 → Target_Risco_SLA=0 (isento)
└─ INC-2025-CHILD-2 (filho) → Possui_Pai=1 → Target_Risco_SLA=0 (isento)
```

---

### 4️⃣ Feature: `Duracao_Horas` (Numeric)

**Pergunta**: "Quanto tempo (em horas) levou para resolver este incidente?"

**Lógica**:
```python
Duracao_Horas = Duração / 3600  # Conversão de segundos para horas
```

**Uso em ML**:
- Preditor de complexidade
- Comparação com SLA (se Duracao_Horas > SLA_horas, violação)

**Estatísticas Esperadas**:
```
P1: média ~2 horas (SLA=4h)
P5: média ~30 horas (SLA=120h)
```

---

### 5️⃣ Feature: `Data_Abertura` (Date)

**Pergunta**: "Qual foi a data (sem hora) de abertura?"

**Lógica**:
```python
Data_Abertura = Aberto.date()  # Extrai apenas YYYY-MM-DD
```

**Uso em ML**:
- Grouping para agregações (volume diário)
- Feature temporal para Prophet (seasonality)
- Day-of-week (segunda=violação maior que sexta?)

---

### 6️⃣ Feature: `KPI_Status_Int` (Integer com Codificação Especial)

**Pergunta**: "Qual é o status de compliance de SLA do incidente?"

**Lógica**:
```python
KPI_Status_Int = {
    'SIM':  1,      # SLA violado
    'NAO':  0,      # SLA respeitado
    None: -1        # Desconhecido (crítico!)
}
```

**Por que -1 para nulo?**

🔑 **Situação**: KPI_Violado é nulo para ~15% dos incidentes. Por quê?

- Incidentes abertos em 2025 mas não fechados até today → sem resultado KPI
- Incidentes com Status='Encerrado' sem data Resolvido → KPI indeterminado

**Decisão**: Usar -1 como "sentinela" para marcar incerteza, e depois corrigir via lógica de negócio:

```python
# Heurística: Se P2, Duração > 4h, e KPI_Status_Int = -1, marca como violado
if KPI_Status_Int == -1 and Prioridade_Num == 2 and Duracao_Horas > 4:
    Target_Risco_SLA = 1  # Provavelmente violado (heurística)
```

## 🧹 Fase 5: Tratamento de Nulos - Regras de Negócio

### Princípio: "Nulos têm significado de negócio"

❌ **Erro comum**: Deletar registros com nulos ou imputar com média  
✅ **Correto**: Substituir com valor que representa a realidade operacional

### Mapeamento de Regras

```python
FILLNA_RULES = {
    'Produto': 'Não Classificado',        # Se nulo → sem produto associado
    'Categoria': 'Não Classificado',      # Se nulo → sem categoria
    'Subcategoria': 'Não Informada',      # Se nulo → não informado (mais específico)
    'Incidente_Pai': 'Independente',      # Se nulo → não tem pai (é independente)
    'Código_de_fechamento': 'Não Encerrado',  # Se nulo → ainda em progresso
    'Solução': 'Sem Descrição',           # Se nulo → resolução sem documentação
}
```

### Por que cada valor?

| Campo | Nulo = ? | Significado | ML Impact |
|-------|----------|-------------|----------|
| `Produto` | Produto desconhecido | Sistema incorretamente classificado | Cluster separado ("Unknown") |
| `Categoria` | Sem categoria | Incidente genérico | Variância aumenta |
| `Subcategoria` | Não informado | Documentação incompleta | OK (menos granulado) |
| `Incidente_Pai` | Independente | Não vinculado (é root) | Facilita análise (não filha) |
| `Código_de_fechamento` | Sem fechamento | Incidente em aberto | Alertar (pode estar pendurado) |
| `Solução` | Sem documentação | Problema desconhecido | Reduz interpretabilidade |

### Dados que NÃO preenchemos

```python
# Mantemos como NaT (Null as Type):
# - Resolvido (if null → incidente não foi resolvido)
# - Encerrado (if null → não fechado formalmente)
# 
# Razão: Informação temporal é crítica.
# Preencher com data arbitrária = viés de duração
```

## 🎲 Fase 6: Calculando o Target para ML - `Target_Risco_SLA`

### Pergunta Central

**"Qual é a probabilidade deste incidente violar seu SLA?"**

### Algoritmo de 3 Camadas

```
Camada 1: Base (KPI_Status_Int)
Camada 2: Heurística (se KPI desconhecido, deduzir)
Camada 3: Isenções (isentar tipos que não contam)
```

#### Camada 1: Usar KPI_Violado Diretamente

```python
if KPI_Status_Int == 1:    # 'SIM'
    Target_Risco_SLA = 1   # Violado
elif KPI_Status_Int == 0:  # 'NAO'
    Target_Risco_SLA = 0   # Respeitado
```

**Cobertura**: ~85% dos incidentes

#### Camada 2: Heurística para KPI Desconhecido (-1)

```python
# Cenário 1: P2 com duração > SLA
if KPI_Status_Int == -1 and Prioridade_Num == 2 and Duracao_Horas > 4:
    Target_Risco_SLA = 1   # Provavelmente violado

# Cenário 2: P3 no fim de semana (8x5 SLA pause)
# Se dia_semana in [5, 6] (sexta/sábado):
#   SLA_esperado = 5 dias (não conta fim de semana)
#   Mas Duração conta tudo (122 horas de calendario)
#   → Falso negativo comum
if KPI_Status_Int == -1 and Prioridade_Num == 3 and weekday in [5, 6]:
    # Investigar duração, mas não marcar como violado automaticamente
    Target_Risco_SLA = -1  # Deixar em dúvida (ou skip do treinamento)
```

**Cobertura adicional**: ~10% dos incidentes

#### Camada 3: Isenções Regulatórias

```python
# Isenção 1: Incidente filho não conta (vinculado ao pai)
if Possui_Pai == 1:
    Target_Risco_SLA = 0   # Isento (pai responde)

# Isenção 2: Sem intervenção = não contabiliza
if Exige_Intervencao == 0:
    Target_Risco_SLA = 0   # Isento (ruído)
```

**Efeito**: Reduz noise, melhora qualidade do target

### Resultado Final: Distribuição de Classes

```
Target_Risco_SLA = 0 (Sem Risco):     40.877 registros (98.6%)
Target_Risco_SLA = 1 (Com Risco):       564 registros (1.4%)

Razão de desbalanceamento: 72:1 (desfavorável para XGBoost)
Ação: Usar class_weight='balanced' ou SMOTE durante treino
```

### Por que esse desbalanceamento?

✅ **Realismo operacional**: A maioria dos incidentes é resolvida NO prazo  
✅ **Raro é importante**: Violação é o caso que queremos prever (business value)  
⚠️ **Desafio ML**: Modelo ingênuo (prevê sempre 0) teria 98.6% acurácia mas seria inútil

## ✅ Fase 7: Validações de Qualidade Pós-Processamento

### Princípio: "Confio em medidas, não em esperança"

Antes de escrever Silver, validamos:

### 1. Validação de Schema

```python
expected_cols = {
    'Número': StringType,
    'Prioridade_Num': IntegerType,    # Novo
    'Duracao_Horas': DoubleType,      # Novo
    'Target_Risco_SLA': IntegerType,  # Novo
    ...
}

for col, dtype in expected_cols.items():
    assert col in df.columns, f"Coluna {col} faltando"
    assert str(df[col].dtype) == str(dtype), f"Tipo de {col} incorreto"
```

✅ Garante que a estrutura está correcta

### 2. Validação de Contagem

```python
total = df.count()
print(f"Total de registros: {total}")

assert 35_000 < total < 50_000, f"Contagem anômala: {total}"
# Se Silver tiver < 35k, algo deu errado (filtro muito agressivo)
# Se tiver > 50k, filtro não funcionou (dados históricos inclusos)
```

✅ Detecta corrupção do filtro temporal

### 3. Validação de Features Derivadas

```python
# Exige_Intervencao deve ser 0 ou 1
assert df.filter(~F.col('Exige_Intervencao').isin([0, 1])).count() == 0

# Prioridade_Num deve estar em [1, 5]
assert df.filter((F.col('Prioridade_Num') < 1) | (F.col('Prioridade_Num') > 5)).count() == 0

# Duracao_Horas deve ser positiva (se não nula)
assert df.filter((F.col('Duracao_Horas') < 0)).count() == 0
```

✅ Verifica lógica de feature engineering

### 4. Análise de Nulos Críticos

```python
nulos_por_coluna = {
    'Número': df.filter(F.col('Número').isNull()).count(),
    'Prioridade_Num': df.filter(F.col('Prioridade_Num').isNull()).count(),
    'Target_Risco_SLA': df.filter(F.col('Target_Risco_SLA').isNull()).count(),
}

for col, cnt in nulos_por_coluna.items():
    ratio = 100 * cnt / total
    print(f"{col}: {cnt} nulos ({ratio:.2f}%)")
    
    if col in ['Número', 'Target_Risco_SLA'] and cnt > 0:
        raise ValueError(f"Coluna crítica {col} tem nulos!")
```

✅ Garante que dados críticos estão completos

### 5. Estatísticas do Target (Verificação de Sanidade)

```python
target_dist = df.groupBy('Target_Risco_SLA').count().collect()

for row in target_dist:
    label, count = row[0], row[1]
    pct = 100 * count / total
    print(f"Target={label}: {count} ({pct:.2f}%)")

# Exemplo output:
# Target=0: 40,877 (98.6%)
# Target=1:    564 (1.4%)
```

✅ Confirma desbalanceamento e plano de ação (SMOTE/class_weight)

## 🚀 Fase 8: Performance e Escalabilidade

### Estimativa de Tempo de Execução

```
Input:  122.543 registros (Parquet em S3)
Output: 41.441 registros (Parquet em S3, particionado por ano/mês)

Glue Config:
  Workers: 10 × G.2X (16 vCPU, 122 GB RAM cada)
  Total: 160 vCPU, 1.2 TB RAM

Breakdown estimado:
  Read Bronze:       ~1 min  (ótimo em S3 com Parquet)
  Schema validation: <1 min
  Filtering:        ~2 min  (parallelizável)
  Feature eng:      ~5 min  (CPU-bound, mas paralelizável)
  Null handling:    ~2 min
  Target calc:      ~3 min
  Write Silver:     ~5 min  (escritura em S3, particionação)
  Validation:       ~1 min
  ─────────────────────
  TOTAL: ~20 minutos
```

### Custo Estimado

```
Glue pricing: $0.44 por DPU-hora (no us-east-1)
Job duration: 20 minutos = 0.33 horas
DPUs: 10 workers

Cost = 10 DPU × 0.33 h × $0.44 = $1.45 por execução

Se rodar daily (Airflow):
  Mensal: 30 × $1.45 = $43.50
  Anual:  365 × $1.45 = $529
```

**Interpretação**: Muito barato. Se performance piorar, pode aumentar workers (linear scaling com PySpark).

### Otimizações Possíveis (Roadmap Futuro)

```python
# Hoje: 10 workers (baseline)
# Se dataset crescer 10x: 15 workers
# Se SLA precisa ser < 5 min: 25 workers

# Outras optimizações:
# 1. Usar Parquet com Snappy compression (padrão)
# 2. Particionar por Data_Abertura (E7 - não aqui)
# 3. Usar Spark native SQL via Glue Catalog (E6-E7)
# 4. Caching de DataFrames intermediários (se reusar)
```

## 🔄 Fase 9: Integração com Glue Catalog e dbt (Preview E6-E7)

### Próximo Passo: Silver → Gold

Uma vez que `incidents_silver_2025.parquet` está em S3, as próximas fases usarão:

```
E6 (Silver via dbt):
  ├─ Registrar Silver no Glue Catalog
  ├─ dbt models apontam para S3 como source
  └─ dbt test (unique, not_null, etc)

E7 (Gold + Star Schema):
  ├─ dbt models criam dimensões no RDS (dim_data, dim_prioridade, ...)
  ├─ dbt models criam facts no RDS (fact_incidents_kpi)
  └─ dbt models criam views para BI (vw_painel, vw_alertas)

E8 (ML Models):
  ├─ Prophet: lê fact_incidents_kpi, faz forecast D+1/D+7
  ├─ XGBoost: treina em Gold features, prediz Target_Risco_SLA
  └─ K-Means: clustering de incidentes por padrão
```

### Contrato entre Glue e dbt

```python
# Silver Parquet (output Glue)
s3://aiops-locaweb-datalake-2026/silver/incidents_silver_2025.parquet/
├── Ano_Mes=2025-01/
│   ├── part-00000.parquet
│   ├── part-00001.parquet
│   └── ...
├── Ano_Mes=2025-02/
└── ...

# dbt sources.yml (input dbt)
sources:
  - name: aws_s3
    tables:
      - name: incidents_silver
        external_location: s3://aiops-locaweb-datalake-2026/silver/incidents_silver_2025.parquet/
        format: parquet
        partition_keys: [Ano_Mes]
```

## 📋 Fase 10: Plano de Implementação Detalhado

### Timeline (Sprints)

| Sprint | Data | Tarefa | Responsável | Status |
|--------|------|--------|-------------|--------|
| **Sprint 1** | 2026-05-20 | Criar estrutura de pastas + módulos base | Dev | ⏳ |
| **Sprint 1** | 2026-05-20 | Implementar `transform_bronze_to_silver.py` | Dev | ⏳ |
| **Sprint 2** | 2026-05-22 | Testes unitários (pytest) | QA | ⏳ |
| **Sprint 2** | 2026-05-22 | Testes de integração (local Spark) | QA | ⏳ |
| **Sprint 3** | 2026-05-24 | Upload scripts para S3 | DevOps | ⏳ |
| **Sprint 3** | 2026-05-25 | Terraform apply (criar job Glue) | DevOps | ⏳ |
| **Sprint 4** | 2026-05-26 | Teste end-to-end (Glue job manual) | QA | ⏳ |
| **Sprint 4** | 2026-05-27 | CloudWatch logs + alertas | DevOps | ⏳ |
| **Sprint 5** | 2026-05-28 | Integração com Airflow DAG (E9) | Dev | ⏳ |

### Estrutura de Pastas (O que vamos criar)

```
pipeline/silver/
├── __init__.py
├── transform_bronze_to_silver.py    # Job principal Glue
├── transformations.py               # Lógica reutilizável (features)
├── validators.py                    # Validações de qualidade
├── handlers.py                      # Tratamento de erro
├── logger.py                        # Logging customizado
├── test_transformations.py          # Unit tests
├── test_integration.py              # Integration tests
└── README.md                        # Documentação

infra/terraform/modules/glue/
├── main.tf                          # IAM roles, Glue job
├── variables.tf
├── outputs.tf
└── glue_job_config.json             # Configurações job
```

## 🎓 Fase 11: Lições Aprendidas e Decisões Críticas

### Decisão #1: Por que não fazer tudo em dbt?

❌ **Alternativa**: Ler Bronze no dbt, fazer all transforms no RDS PostgreSQL

✅ **Por que Glue + dbt em conjunto**:
- Glue: Processamento em Spark (paralelismo, I/O em S3)
- dbt: Transformações complexas em SQL (fácil debugging, versionável)
- Separação de responsabilidades: raw → clean (Glue), clean → analytic (dbt)
- dbt é melhor para "star schemas" e dimensões (E7)

### Decisão #2: Por que 41.441 registros e não 122.543?

```
Bronze: 122.543 registros
  └─ Menos: Status='Sem Intervenção' (~81.102 = ruído de monitoramento)
  └─ Menos: Data < 2025-01-01 (~histórico, contexto mudou)
  ─────────────────────
Silver: 41.441 registros (esforço real, 2025+)
```

💡 **Insight**: 67% dos registros ITSM são "falsos positivos" (notificações sem ação humana).

### Decisão #3: Target desbalanceado (98.6% vs 1.4%) é um problema?

✅ **NÃO**. Razões:
1. **Realismo**: Incidentes violados são RAROS (isso é bom!)
2. **Business value**: Queremos prever os RAROS (1.4%)
3. **Métricas certas**: Usar AUC-ROC, F1, não acurácia
4. **Técnicas**: SMOTE ou class_weight='balanced' em XGBoost

### Decisão #4: Por que particionar por Ano_Mes?

```
s3://silver/incidents_silver_2025.parquet/
├── Ano_Mes=2025-01/
├── Ano_Mes=2025-02/
└── ...

Benefícios:
1. Leitura eficiente: dbt filtra por partição (menos I/O)
2. Exclusão de partições: se 2025-01 corromper, reprocessa só ela
3. Escalabilidade: se dados crescerem, partições distribuem carga
```

### Decisão #5: Glue vs Databricks vs Spark local?

| Critério | Glue | Databricks | Spark Local |
|----------|------|-----------|-------------|
| Scaling | ✅ Automático | ✅ Automático | ❌ Manual |
| Custo | ✅ Por uso | ❌ Por cluster | ✅ Só hardware |
| AWS Integration | ✅ Nativo | ⚠️ Cloud-agnostic | ❌ Manual |
| IDE UX | ⚠️ Console | ✅ Excelente | ✅ Notebook |
| **Escolha** | **✅ Escolhido** | ❌ Não | ❌ Não |

💡 **Resultado**: Glue é pragmático para este projeto (AWS-first, serverless, baixo custo).

## 🏁 Conclusão: O Big Picture

### O que construímos?

Um **pipeline automático, monitorado e escalável** que transforma dados brutos de ITSM em features pronta para ML.

```
Input:  122.543 incidentes (2018-2026, incluindo ruído)
Process: Glue job em Apache Spark (10 workers, ~20 min)
Output: 41.441 incidentes limpos + 6 features derivadas + target ML
```

### Por que isto importa?

✅ **Reprodutibilidade**: Mesmos dados de entrada → mesma saída (determinístico)  
✅ **Rastreabilidade**: Cada coluna, cada linha tem origem clara  
✅ **Escalabilidade**: Se crescer para 1M+ incidentes, adiciona workers  
✅ **Monitoramento**: CloudWatch logs, alertas automáticas  
✅ **Integração**: Airflow orquestra tudo (E9)  

### Roadmap Futuro

```
E5 (agora):     Glue Bronze → Silver ✅
E6 (próxima):   dbt Silver → Gold (star schema)
E7 (depois):    Dimensões + Fatos no RDS
E8 (depois):    ML models (Prophet, XGBoost, K-Means)
E9 (depois):    Airflow orquestração
E10 (depois):   Power BI dashboards
```

### Tomar Casa

**"Uma boa solução não é perfeita, é pragmática."**

- ✅ Glue é managed → menos infraestrutura
- ✅ PySpark é comum → time consegue manutenção
- ✅ S3 Parquet é padrão → integra bem com dbt/BI
- ✅ Logging automático → observabilidade de graça

---

**Próximo passo**: Implementar os scripts conforme plano em `md/PLANO_GLUE_SILVER_PARQUET.md`.

## 📚 Referências e Recursos

### Documentação
- [AWS Glue Documentation](https://docs.aws.amazon.com/glue/)
- [PySpark SQL Reference](https://spark.apache.org/docs/latest/api/python/)
- [Medallion Lake Architecture](https://www.databricks.com/blog/2022/06/24/simplify-data-pipelines-with-delta-lake.html)

### Arquivos do Projeto
- [PLANO_GLUE_SILVER_PARQUET.md](../md/PLANO_GLUE_SILVER_PARQUET.md) — Detalhes técnicos
- [CLAUDE.md](../CLAUDE.md) — Setup geral do projeto
- [01_pre_processamento_bronze.ipynb](01_pre_processamento_bronze.ipynb) — Bronze standards
- [02_eda_silver.ipynb](02_eda_silver.ipynb) — Feature engineering original

### Conceitos ML
- **SMOTE**: Synthetic Minority Over-sampling (para desbalanceamento)
- **AUC-ROC**: Area Under the Receiver Operating Characteristic Curve
- **Class Weight**: Penalização de classes minoritárias
- **Feature Importance**: SHAP values (XGBoost explicabilidade)